# 1. Bronze — Ingestion Auto Loader

Ingestion incrémentale des fichiers de sinistres (CSV, séparateur `;`) depuis le volume Unity Catalog `workspace.default.projet_alpha` vers la table Delta `workspace.projet_alpha.insurance_claim_fraud_bronze`.

**Auto Loader** + `trigger(availableNow=True)` : chaque exécution (manuelle ou planifiée) ne traite que les nouveaux fichiers détectés grâce au checkpoint — le notebook est idempotent et prêt pour un job quotidien.

In [0]:
# Paramètres du pipeline Bronze
source_dir = "/Volumes/workspace/default/projet_alpha"   # dossier surveillé par Auto Loader
bronze_table = "workspace.projet_alpha.insurance_claim_fraud_bronze"
checkpoint_path = "/Volumes/workspace/default/projet_alpha/_checkpoints/bronze_ingestion"
schema_location = "/Volumes/workspace/default/projet_alpha/_checkpoints/bronze_schema"

# Reset optionnel : True = repartir de zéro (drop table + checkpoint vidé), à utiliser UNE FOIS
RESET_BRONZE = False

if RESET_BRONZE:
    spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
    dbutils.fs.rm(checkpoint_path, True)
    print("Bronze et checkpoint réinitialisés.")

In [0]:
# Ingestion Auto Loader : CSV (';' + header) -> table Delta bronze, avec colonnes de traçabilité
from pyspark.sql.functions import col, current_timestamp

(spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", schema_location)
  .option("header", "true")
  .option("sep", ";")
  .option("cloudFiles.inferColumnTypes", "true")      # types inférés
  .load(source_dir)
  .select("*",
          col("_metadata.file_path").alias("source_file"),
          col("_metadata.file_modification_time").alias("source_file_modification_time"),
          current_timestamp().alias("bronze_ingestion_timestamp"))
  .writeStream
  .option("checkpointLocation", checkpoint_path)     # mémorise les fichiers déjà traités -> pas de doublon au re-run
  .trigger(availableNow=True)                          # traite les fichiers en attente puis s'arrête
  .toTable(bronze_table))

In [0]:
# Contrôle post-ingestion : volume bronze et fichiers sources distincts
bronze_df = spark.table(bronze_table)
nb_rows = bronze_df.count()
nb_files = bronze_df.select("source_file").distinct().count()
print(f"Table {bronze_table} : {nb_rows} lignes issues de {nb_files} fichier(s) source.")
display(bronze_df.limit(10))

In [0]:
# Schéma de la table bronze
bronze_df.printSchema()

## EDA — claim_amount par démographie

Exploration initiale menée sur la table bronze : répartition des montants de réclamation (`claim_amount`) selon les caractéristiques démographiques des assurés (âge, genre, statut marital, profession, revenu, éducation, ancienneté × score de crédit), puis selon le contexte du sinistre (type d'incident × sévérité, lieu, véhicule). Ces analyses exploratoires précèdent la construction de la table Gold (notebook 3).

In [0]:
# Référence SQL de la table bronze pour les requêtes EDA ci-dessous
table_name = bronze_table

# claim_amount moyen par âge (montants quasi uniformes, ~6 800 à ~7 500 $ : pas de surreprésentation d'une classe d'âge)
avg_claim_amount_per_age = spark.sql(f"""
    SELECT age, round(avg(claim_amount), 2) AS avg_claim_amount_dollars
    FROM {table_name}
    GROUP BY age
    ORDER BY age DESC
""")
display(avg_claim_amount_per_age)

In [0]:
# claim_amount total par genre (Female et Male dominent simplement par leur effectif : ~340 M$ chacun)
total_claim_amount_per_gender = spark.sql(f"""
    SELECT gender, round(sum(claim_amount), 2) AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY gender
    ORDER BY total_claim_amount_dollars DESC
""")
display(total_claim_amount_per_gender)

In [0]:
# claim_amount total par statut marital (quasi équirépartition : ~174-178 M$ par statut)
total_claim_amount_per_marital_status = spark.sql(f"""
    SELECT marital_status, round(sum(claim_amount), 2) AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY marital_status
    ORDER BY total_claim_amount_dollars DESC
""")
display(total_claim_amount_per_marital_status)

In [0]:
# claim_amount total par profession (montants très proches d'une profession à l'autre : 48-52 M$)
total_claim_amount_per_occupation = spark.sql(f"""
    SELECT occupation, round(sum(claim_amount), 2) AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY occupation
    ORDER BY total_claim_amount_dollars DESC
""")
display(total_claim_amount_per_occupation)

In [0]:
# claim_amount par tranche de revenu annuel (montant moyen ~6 900-7 100 $ quelle que soit la tranche :
# le revenu n'influence pas le montant réclamé ; les totaux suivent juste les effectifs)
total_claim_amount_per_income_bracket = spark.sql(f"""
    SELECT CASE
               WHEN annual_income < 30000 THEN '<30k'
               WHEN annual_income < 60000 THEN '30k-60k'
               WHEN annual_income < 90000 THEN '60k-90k'
               ELSE '90k+'
           END AS income_bracket,
           replace(format_number(avg(annual_income), 2), ',', ' ') AS avg_annual_income,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars,
           round(avg(claim_amount), 2) AS avg_claim_amount_dollars,
           count(*) AS nb_claims
    FROM {table_name}
    GROUP BY income_bracket
    ORDER BY avg_annual_income
""")
display(total_claim_amount_per_income_bracket)

In [0]:
# claim_amount total par niveau d'éducation (équirépartition stricte : ~139-144 M$ par niveau)
total_claim_amount_per_education_level = spark.sql(f"""
    SELECT education_level,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY education_level
    ORDER BY total_claim_amount_dollars DESC
""")
display(total_claim_amount_per_education_level)

In [0]:
# Vue croisée : seniorité dans la compagnie x appréciation du score de crédit
# (durée moyenne de traitement ~22-23 jours constante ; les totaux suivent les effectifs,
# aucune combinaison démographique ne ressort)
seniority_level_credit_score_claim_amount = spark.sql(f"""
    SELECT CASE
               WHEN years_with_company < 2 THEN 'Junior'
               WHEN years_with_company < 5 THEN 'Mid-level'
               WHEN years_with_company < 10 THEN 'Senior'
               ELSE 'Expert'
           END AS seniority_level_company,
           CASE
               WHEN credit_score < 579 THEN 'Bad'
               WHEN credit_score > 580 AND credit_score < 669 THEN 'Medium'
               WHEN credit_score > 670 AND credit_score < 739 THEN 'Good'
               WHEN credit_score > 740 AND credit_score < 799 THEN 'Very good'
               WHEN credit_score > 800 AND credit_score < 850 THEN 'Excellent'
               ELSE 'Excellent +'
           END AS credit_score_appreciation,
           round(avg(years_with_company), 2) AS avg_years_with_company,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars,
           round(avg(claim_duration_days), 2) AS avg_claim_duration_days,
           count(*) AS nb_claims
    FROM {table_name}
    GROUP BY seniority_level_company, credit_score_appreciation
    ORDER BY seniority_level_company DESC, nb_claims DESC
""")
display(seniority_level_credit_score_claim_amount)

In [0]:
# claim_amount total par type d'incident x sévérité
# (ordre de grandeur homogène par sévérité : Minor ~9-10 M$, Moderate ~26-27 M$, Major ~40-44 M$, Total Loss ~37-40 M$
# — la sévérité porte le montant, pas le type d'incident)
claim_amount_by_incident_type_severity = spark.sql(f"""
    SELECT incident_type,
           accident_severity,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY incident_type, accident_severity
    ORDER BY accident_severity DESC, sum(claim_amount) DESC
""")
display(claim_amount_by_incident_type_severity)

In [0]:
# claim_amount total par lieu de l'incident x sévérité
# (mêmes ordres de grandeur partout : la géographie du dataset est neutre sur les montants)
claim_amount_per_incident_location = spark.sql(f"""
    SELECT incident_location,
           accident_severity,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY accident_severity, incident_location
    ORDER BY sum(claim_amount) DESC, accident_severity DESC
""")
display(claim_amount_per_incident_location)

In [0]:
# Véhicules : claim_amount par catégorie d'âge, dommage, marque et modèle — puis requêtes de détail
# (les 7 tables se lisent de l'agrégat le plus large au plus fin pour comprendre ce qui se cache derrière la grande requête)
claim_amount_per_vehicle_profile = spark.sql(f"""
    SELECT CASE
                WHEN vehicle_age < 1 THEN 'Zero km'
                WHEN vehicle_age < 2 THEN 'Recent pre-owned'
                WHEN vehicle_age < 3 THEN 'Core of market'
                WHEN vehicle_age < 4 THEN 'End of leasing cycle'
                WHEN vehicle_age < 5 THEN 'Standard pre-owned item'
                WHEN vehicle_age < 10 THEN 'Low-end / Second-rate vehicle'
                WHEN vehicle_age < 15 THEN 'Budget-friendly second-hand option'
                ELSE 'Collector vehicle'
            END AS vehicle_age_category,
        vehicle_damage,
        vehicle_make,
        vehicle_model,
        replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY vehicle_damage, vehicle_make, vehicle_model, vehicle_age_category
    ORDER BY sum(claim_amount) DESC, vehicle_age_category DESC
""")
display(claim_amount_per_vehicle_profile)
displayHTML("<h3>Table 1 : Montant total des réclamations par catégorie d'âge du véhicule, dommage, marque et modèle</h3>")

# 1) Vue agrégée par catégorie d'âge du véhicule uniquement
claim_amount_per_vehicle_age_category = spark.sql(f"""
    SELECT CASE
               WHEN vehicle_age < 1 THEN 'Zero km'
               WHEN vehicle_age < 2 THEN 'Recent pre-owned'
               WHEN vehicle_age < 3 THEN 'Core of market'
               WHEN vehicle_age < 4 THEN 'End of leasing cycle'
               WHEN vehicle_age < 5 THEN 'Standard pre-owned item'
               WHEN vehicle_age < 10 THEN 'Low-end / Second-rate vehicle'
               WHEN vehicle_age < 15 THEN 'Budget-friendly second-hand option'
               ELSE 'Collector vehicle'
           END AS vehicle_age_category,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars,
           round(avg(claim_amount), 2) AS avg_claim_amount_dollars,
           count(*) AS nb_claims
    FROM {table_name}
    GROUP BY vehicle_age_category
    ORDER BY total_claim_amount_dollars DESC
""")
display(claim_amount_per_vehicle_age_category)
displayHTML("<h3>Table 2 : Réclamations par catégorie d'âge du véhicule uniquement</h3>")

# 2) Vue agrégée par niveau de dommage du véhicule
claim_amount_per_vehicle_damage = spark.sql(f"""
    SELECT vehicle_damage,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars,
           round(avg(claim_amount), 2) AS avg_claim_amount_dollars,
           count(*) AS nb_claims
    FROM {table_name}
    GROUP BY vehicle_damage
    ORDER BY nb_claims DESC
""")
display(claim_amount_per_vehicle_damage)
displayHTML("<h3>Table 3 : Réclamations par niveau de dommage du véhicule</h3>")

# 3) Vue croisée : catégorie d'âge x dommage du véhicule
claim_amount_per_age_category_damage = spark.sql(f"""
    SELECT CASE
               WHEN vehicle_age < 1 THEN 'Zero km'
               WHEN vehicle_age < 2 THEN 'Recent pre-owned'
               WHEN vehicle_age < 3 THEN 'Core of market'
               WHEN vehicle_age < 4 THEN 'End of leasing cycle'
               WHEN vehicle_age < 5 THEN 'Standard pre-owned item'
               WHEN vehicle_age < 10 THEN 'Low-end / Second-rate vehicle'
               WHEN vehicle_age < 15 THEN 'Budget-friendly second-hand option'
               ELSE 'Collector vehicle'
           END AS vehicle_age_category,
           vehicle_damage,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars,
           round(avg(claim_amount), 2) AS avg_claim_amount_dollars,
           count(*) AS nb_claims
    FROM {table_name}
    GROUP BY vehicle_age_category, vehicle_damage
    ORDER BY vehicle_age_category, vehicle_damage
""")
display(claim_amount_per_age_category_damage)
displayHTML("<h3>Table 4 : Vue croisée — catégorie d'âge x niveau de dommage</h3>")

# 4) Top marques/modèles par montant total de réclamation
claim_amount_per_vehicle_make_model = spark.sql(f"""
    SELECT vehicle_make,
           vehicle_model,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars,
           round(avg(claim_amount), 2) AS avg_claim_amount_dollars,
           count(*) AS nb_claims
    FROM {table_name}
    GROUP BY vehicle_make, vehicle_model
    ORDER BY total_claim_amount_dollars DESC
    LIMIT 50
""")
display(claim_amount_per_vehicle_make_model)
displayHTML("<h3>Table 5 : Top marques/modèles par montant total de réclamations</h3>")

# 5) Distribution de l'âge exact des véhicules (pour comprendre les catégories)
vehicle_age_distribution = spark.sql(f"""
    SELECT vehicle_age,
           count(*) AS nb_claims,
           round(avg(claim_amount), 2) AS avg_claim_amount_dollars,
           replace(format_number(sum(claim_amount), 2), ',', ' ') AS total_claim_amount_dollars
    FROM {table_name}
    GROUP BY vehicle_age
    ORDER BY vehicle_age
""")
display(vehicle_age_distribution)
displayHTML("<h3>Table 6 : Distribution de l'âge exact des véhicules</h3>")

# 6) Détail des plus grosses réclamations individuelles (lignes derrière les agrégats)
top_individual_claims = spark.sql(f"""
    SELECT vehicle_age,
           vehicle_damage,
           vehicle_make,
           vehicle_model,
           claim_amount,
           incident_type,
           accident_severity
    FROM {table_name}
    ORDER BY claim_amount DESC
    LIMIT 50
""")
display(top_individual_claims)
displayHTML("<h3>Table 7 : Top 50 des plus grosses réclamations individuelles</h3>")